# Edge Device Profiling Notebook

Profile and optimize inference on NVIDIA Jetson devices.

## Contents
1. Device Information
2. Power Mode Analysis
3. Memory Profiling
4. Thermal Monitoring
5. Multi-Stream Capacity
6. Optimization Recommendations

In [ ]:
# Standard imports
import sys
import time
import subprocess
from pathlib import Path
from typing import Dict, List, Any, Optional
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Add project root
sys.path.insert(0, str(Path.cwd().parent))

print("Environment ready!")

## 1. Device Information

In [ ]:
@dataclass
class DeviceInfo:
    """Jetson device information."""
    model: str
    l4t_version: str
    jetpack_version: str
    cuda_version: str
    tensorrt_version: str
    gpu_name: str
    gpu_memory_mb: int
    cpu_cores: int
    ram_mb: int

def get_device_info() -> DeviceInfo:
    """Get Jetson device information (mock for non-Jetson systems)."""
    
    # Try to get real info
    try:
        with open('/etc/nv_tegra_release') as f:
            l4t = f.read().strip()
    except FileNotFoundError:
        l4t = "R35.3.1"  # Mock
    
    # Mock device info for demo
    return DeviceInfo(
        model="NVIDIA Jetson Orin NX 16GB",
        l4t_version=l4t,
        jetpack_version="5.1.2",
        cuda_version="11.4",
        tensorrt_version="8.5.2",
        gpu_name="NVIDIA Orin (8 SM)",
        gpu_memory_mb=16384,
        cpu_cores=8,
        ram_mb=16384,
    )

device = get_device_info()

print("="*60)
print("DEVICE INFORMATION")
print("="*60)
print(f"Model:           {device.model}")
print(f"L4T Version:     {device.l4t_version}")
print(f"JetPack:         {device.jetpack_version}")
print(f"CUDA:            {device.cuda_version}")
print(f"TensorRT:        {device.tensorrt_version}")
print(f"GPU:             {device.gpu_name}")
print(f"GPU Memory:      {device.gpu_memory_mb} MB")
print(f"CPU Cores:       {device.cpu_cores}")
print(f"System RAM:      {device.ram_mb} MB")

## 2. Power Mode Analysis

In [ ]:
# Define power modes for different Jetson devices
POWER_MODES = {
    'Jetson Orin NX': [
        {'id': 0, 'name': 'MAXN', 'power_w': 25, 'gpu_freq_mhz': 918, 'cpu_freq_mhz': 2000},
        {'id': 1, 'name': '15W', 'power_w': 15, 'gpu_freq_mhz': 612, 'cpu_freq_mhz': 1500},
        {'id': 2, 'name': '10W', 'power_w': 10, 'gpu_freq_mhz': 510, 'cpu_freq_mhz': 1200},
    ],
    'Jetson AGX Orin': [
        {'id': 0, 'name': 'MAXN', 'power_w': 60, 'gpu_freq_mhz': 1300, 'cpu_freq_mhz': 2200},
        {'id': 1, 'name': '50W', 'power_w': 50, 'gpu_freq_mhz': 1100, 'cpu_freq_mhz': 2000},
        {'id': 2, 'name': '30W', 'power_w': 30, 'gpu_freq_mhz': 900, 'cpu_freq_mhz': 1500},
        {'id': 3, 'name': '15W', 'power_w': 15, 'gpu_freq_mhz': 612, 'cpu_freq_mhz': 1200},
    ],
    'Jetson Orin Nano': [
        {'id': 0, 'name': '15W', 'power_w': 15, 'gpu_freq_mhz': 625, 'cpu_freq_mhz': 1500},
        {'id': 1, 'name': '7W', 'power_w': 7, 'gpu_freq_mhz': 306, 'cpu_freq_mhz': 1200},
    ],
}

# Mock performance by power mode
def simulate_power_mode_performance(power_w: int, base_fps: float = 200) -> Dict:
    """Simulate performance at different power levels."""
    # Performance roughly scales with power
    efficiency = min(1.0, power_w / 25)  # Normalize to 25W
    fps = base_fps * efficiency
    latency = 1000 / fps
    
    return {
        'fps': fps,
        'latency_ms': latency,
        'efficiency_fps_per_w': fps / power_w,
    }

# Analyze power modes
modes = POWER_MODES.get('Jetson Orin NX', [])
power_analysis = []

for mode in modes:
    perf = simulate_power_mode_performance(mode['power_w'])
    power_analysis.append({
        'mode': mode['name'],
        'power_w': mode['power_w'],
        'gpu_freq_mhz': mode['gpu_freq_mhz'],
        **perf,
    })

power_df = pd.DataFrame(power_analysis)
print("\nPower Mode Analysis:")
print(power_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Performance vs Power
ax1 = axes[0]
ax1.bar(power_df['mode'], power_df['fps'], color='steelblue', edgecolor='black')
ax1.set_xlabel('Power Mode')
ax1.set_ylabel('FPS')
ax1.set_title('Throughput by Power Mode')
for i, (mode, fps) in enumerate(zip(power_df['mode'], power_df['fps'])):
    ax1.text(i, fps + 5, f'{fps:.0f}', ha='center')

# Latency vs Power
ax2 = axes[1]
ax2.bar(power_df['mode'], power_df['latency_ms'], color='coral', edgecolor='black')
ax2.set_xlabel('Power Mode')
ax2.set_ylabel('Latency (ms)')
ax2.set_title('Latency by Power Mode')
for i, (mode, lat) in enumerate(zip(power_df['mode'], power_df['latency_ms'])):
    ax2.text(i, lat + 0.2, f'{lat:.1f}', ha='center')

# Efficiency
ax3 = axes[2]
ax3.bar(power_df['mode'], power_df['efficiency_fps_per_w'], color='seagreen', edgecolor='black')
ax3.set_xlabel('Power Mode')
ax3.set_ylabel('FPS per Watt')
ax3.set_title('Power Efficiency')
for i, (mode, eff) in enumerate(zip(power_df['mode'], power_df['efficiency_fps_per_w'])):
    ax3.text(i, eff + 0.2, f'{eff:.1f}', ha='center')

plt.tight_layout()
plt.savefig('power_mode_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Memory Profiling

In [ ]:
def simulate_memory_usage(num_streams: int, model_size_mb: int = 50) -> Dict:
    """Simulate memory usage for multi-stream processing."""
    
    # Base memory usage
    system_overhead_mb = 500
    tensorrt_overhead_mb = 200
    
    # Per-stream memory
    frame_buffer_mb = 10  # 1080p frame ~6MB, plus processing buffers
    tracking_buffer_mb = 5
    
    # Calculate totals
    model_memory = model_size_mb
    stream_memory = num_streams * (frame_buffer_mb + tracking_buffer_mb)
    total_gpu_memory = model_memory + tensorrt_overhead_mb + stream_memory
    
    return {
        'num_streams': num_streams,
        'model_mb': model_memory,
        'tensorrt_overhead_mb': tensorrt_overhead_mb,
        'stream_buffers_mb': stream_memory,
        'total_gpu_mb': total_gpu_memory,
        'available_mb': device.gpu_memory_mb - total_gpu_memory,
    }

# Analyze memory for different stream counts
stream_counts = [1, 2, 4, 8, 16, 32]
memory_analysis = [simulate_memory_usage(n) for n in stream_counts]
memory_df = pd.DataFrame(memory_analysis)

print("Memory Usage by Stream Count:")
print(memory_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Stacked bar chart
x = range(len(stream_counts))
width = 0.6

# Stack components
bars1 = ax.bar(x, memory_df['model_mb'], width, label='Model', color='steelblue')
bars2 = ax.bar(x, memory_df['tensorrt_overhead_mb'], width, 
               bottom=memory_df['model_mb'], label='TensorRT Overhead', color='coral')
bars3 = ax.bar(x, memory_df['stream_buffers_mb'], width,
               bottom=memory_df['model_mb'] + memory_df['tensorrt_overhead_mb'],
               label='Stream Buffers', color='seagreen')

# Add total GPU memory line
ax.axhline(y=device.gpu_memory_mb, color='red', linestyle='--', 
           linewidth=2, label=f'GPU Memory Limit ({device.gpu_memory_mb} MB)')

ax.set_xlabel('Number of Streams')
ax.set_ylabel('Memory (MB)')
ax.set_title('GPU Memory Usage by Stream Count')
ax.set_xticks(x)
ax.set_xticklabels(stream_counts)
ax.legend(loc='upper left')

# Add annotations
for i, row in memory_df.iterrows():
    ax.text(i, row['total_gpu_mb'] + 100, f"{row['total_gpu_mb']} MB", 
            ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('memory_usage_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Find max streams
max_streams = memory_df[memory_df['available_mb'] > 0]['num_streams'].max()
print(f"\n✅ Maximum recommended streams: {max_streams}")

## 4. Thermal Monitoring

In [ ]:
def simulate_thermal_profile(duration_seconds: int = 300, 
                            workload: str = 'high') -> pd.DataFrame:
    """Simulate thermal behavior over time."""
    
    # Base temperatures
    ambient = 25
    thermal_resistance = {'high': 0.08, 'medium': 0.05, 'low': 0.03}
    power_draw = {'high': 25, 'medium': 15, 'low': 10}
    
    # Thermal model parameters
    R = thermal_resistance[workload]
    P = power_draw[workload]
    C = 50  # Thermal capacitance
    
    timestamps = np.arange(0, duration_seconds, 1)
    temperatures = []
    
    T = ambient
    for t in timestamps:
        # Simple thermal model with noise
        T_steady = ambient + P * R * 100  # Steady state
        tau = R * C
        T = T_steady - (T_steady - ambient) * np.exp(-t / tau)
        T += np.random.normal(0, 0.5)  # Add noise
        temperatures.append(T)
    
    return pd.DataFrame({
        'time_s': timestamps,
        'temperature_c': temperatures,
        'workload': workload,
    })

# Simulate thermal profiles
thermal_high = simulate_thermal_profile(300, 'high')
thermal_medium = simulate_thermal_profile(300, 'medium')
thermal_low = simulate_thermal_profile(300, 'low')

print("Thermal profile simulation complete.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Temperature over time
ax1 = axes[0]
ax1.plot(thermal_high['time_s'], thermal_high['temperature_c'], 
         label='High Workload (25W)', color='red', linewidth=2)
ax1.plot(thermal_medium['time_s'], thermal_medium['temperature_c'], 
         label='Medium Workload (15W)', color='orange', linewidth=2)
ax1.plot(thermal_low['time_s'], thermal_low['temperature_c'], 
         label='Low Workload (10W)', color='green', linewidth=2)

# Thermal throttle line
ax1.axhline(y=85, color='darkred', linestyle='--', label='Throttle Threshold (85°C)')
ax1.axhline(y=70, color='orange', linestyle='--', alpha=0.5, label='Warning (70°C)')

ax1.set_xlabel('Time (seconds)')
ax1.set_ylabel('Temperature (°C)')
ax1.set_title('GPU Temperature Over Time')
ax1.legend(loc='lower right')
ax1.set_ylim(20, 100)

# Steady state temperatures
ax2 = axes[1]
workloads = ['Low (10W)', 'Medium (15W)', 'High (25W)']
steady_temps = [
    thermal_low['temperature_c'].iloc[-10:].mean(),
    thermal_medium['temperature_c'].iloc[-10:].mean(),
    thermal_high['temperature_c'].iloc[-10:].mean(),
]

colors = ['green', 'orange', 'red']
bars = ax2.bar(workloads, steady_temps, color=colors, edgecolor='black')
ax2.axhline(y=85, color='darkred', linestyle='--', label='Throttle')
ax2.set_ylabel('Steady State Temperature (°C)')
ax2.set_title('Steady State Temperature by Workload')
ax2.set_ylim(0, 100)

for bar, temp in zip(bars, steady_temps):
    ax2.text(bar.get_x() + bar.get_width()/2, temp + 2, f'{temp:.1f}°C', ha='center')

plt.tight_layout()
plt.savefig('thermal_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Multi-Stream Capacity Analysis

In [ ]:
def analyze_stream_capacity(base_fps_per_stream: float = 30,
                           model_latency_ms: float = 5) -> pd.DataFrame:
    """Analyze multi-stream processing capacity."""
    
    # Maximum throughput limited by GPU
    max_gpu_fps = 1000 / model_latency_ms
    
    stream_counts = [1, 2, 4, 8, 16, 32, 48, 64]
    results = []
    
    for num_streams in stream_counts:
        required_fps = num_streams * base_fps_per_stream
        achievable = min(required_fps, max_gpu_fps)
        fps_per_stream = achievable / num_streams
        
        # Check if sustainable
        sustainable = fps_per_stream >= (base_fps_per_stream * 0.95)
        
        results.append({
            'streams': num_streams,
            'required_fps': required_fps,
            'achievable_fps': achievable,
            'fps_per_stream': fps_per_stream,
            'latency_ms': 1000 / fps_per_stream if fps_per_stream > 0 else float('inf'),
            'sustainable': sustainable,
            'gpu_utilization': min(100, (required_fps / max_gpu_fps) * 100),
        })
    
    return pd.DataFrame(results)

capacity_df = analyze_stream_capacity(base_fps_per_stream=30, model_latency_ms=5)
print("Multi-Stream Capacity Analysis:")
print(capacity_df.to_string(index=False))

# Find optimal stream count
max_sustainable = capacity_df[capacity_df['sustainable']]['streams'].max()
print(f"\n✅ Maximum sustainable streams @ 30 FPS: {max_sustainable}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# FPS per stream vs stream count
ax1 = axes[0]
colors = ['green' if s else 'red' for s in capacity_df['sustainable']]
ax1.bar(capacity_df['streams'].astype(str), capacity_df['fps_per_stream'], 
        color=colors, edgecolor='black')
ax1.axhline(y=30, color='blue', linestyle='--', label='Target (30 FPS)')
ax1.set_xlabel('Number of Streams')
ax1.set_ylabel('FPS per Stream')
ax1.set_title('Per-Stream Throughput')
ax1.legend()

# GPU utilization
ax2 = axes[1]
ax2.plot(capacity_df['streams'], capacity_df['gpu_utilization'], 
         marker='o', linewidth=2, markersize=8, color='purple')
ax2.axhline(y=100, color='red', linestyle='--', label='100% Utilization')
ax2.axhline(y=80, color='orange', linestyle='--', alpha=0.5, label='Target (80%)')
ax2.fill_between(capacity_df['streams'], 0, capacity_df['gpu_utilization'], alpha=0.3)
ax2.set_xlabel('Number of Streams')
ax2.set_ylabel('GPU Utilization (%)')
ax2.set_title('GPU Utilization vs Stream Count')
ax2.legend()
ax2.set_ylim(0, 120)

plt.tight_layout()
plt.savefig('stream_capacity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Optimization Recommendations

In [ ]:
print("="*70)
print("EDGE DEPLOYMENT OPTIMIZATION RECOMMENDATIONS")
print("="*70)

print(f"""
🔧 DEVICE: {device.model}
   GPU Memory: {device.gpu_memory_mb} MB
   
📊 CAPACITY SUMMARY:
   • Maximum streams @ 30 FPS: {max_sustainable}
   • Recommended streams (80% utilization): {int(max_sustainable * 0.8)}
   • Memory headroom: {memory_df.iloc[0]['available_mb']} MB (single stream)

⚡ POWER MODE RECOMMENDATIONS:
   • Production (24/7): 15W mode - best efficiency, sustainable thermals
   • Peak Performance: MAXN mode - maximum throughput, requires cooling
   • Battery/Solar: 10W mode - lowest power, reduced performance

🌡️ THERMAL MANAGEMENT:
   • Add heatsink/fan for sustained high workloads
   • Throttle threshold: 85°C
   • Target operating temp: <70°C for longevity
   • Consider active cooling for >8 streams

💾 MEMORY OPTIMIZATION:
   • Use FP16 models (50% memory vs FP32)
   • Enable frame dropping if memory constrained
   • Share model weights across streams (DeepStream)
   • Use batch inference for better efficiency

🚀 PERFORMANCE TIPS:
   1. Use TensorRT with FP16 precision
   2. Enable jetson_clocks for max frequency
   3. Use NvDCF tracker (GPU-accelerated)
   4. Batch multiple frames when possible
   5. Use NVDEC for hardware video decode
   6. Pin memory for faster transfers

📋 RECOMMENDED CONFIGURATION:
   Model: YOLOv8n-FP16
   Batch size: 8
   Input resolution: 640x640
   Power mode: 15W (balanced)
   Streams: {int(max_sustainable * 0.8)} @ 30 FPS
""")

## 7. Export Profiling Report

In [ ]:
import json

# Compile profiling report
report = {
    'device_info': {
        'model': device.model,
        'gpu_memory_mb': device.gpu_memory_mb,
        'jetpack': device.jetpack_version,
        'tensorrt': device.tensorrt_version,
    },
    'power_analysis': power_df.to_dict('records'),
    'memory_analysis': memory_df.to_dict('records'),
    'capacity_analysis': capacity_df.to_dict('records'),
    'recommendations': {
        'max_streams_30fps': int(max_sustainable),
        'recommended_streams': int(max_sustainable * 0.8),
        'recommended_power_mode': '15W',
        'recommended_model': 'YOLOv8n-FP16',
    },
    'generated_at': time.strftime('%Y-%m-%d %H:%M:%S'),
}

with open('edge_profiling_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print("Profiling report exported to edge_profiling_report.json")

# Also export summary CSV
capacity_df.to_csv('stream_capacity_summary.csv', index=False)
print("Capacity summary exported to stream_capacity_summary.csv")